
# 01. Optimizer(최적화) 기법 비교

## 0. 작업 전 준비사항
이 실습은 작은 데이터를 사용하므로 CPU로 충분합니다. 아래 코드는 모델과 텐서를 CPU에 두며, CUDA를 사용할 수 있다는 사실만으로 GPU에서 실행되지는 않습니다.

GPU 환경을 확인하고 싶다면 다음 항목을 점검합니다.
- [ ] GPU 모델 및 드라이버 버전 확인 (`nvidia-smi`, NVIDIA GPU 환경)
- [ ] PyTorch 버전 확인 (`torch.__version__`)
- [ ] 설치된 PyTorch의 CUDA 빌드 버전 확인 (`torch.version.cuda`, CPU 빌드는 `None`)
- [ ] PyTorch의 CUDA 사용 가능 여부 확인 (`torch.cuda.is_available()`)

GPU로 학습하려면 모델과 입력·정답 텐서를 모두 같은 장치로 이동해야 합니다. 이번 비교에서는 CPU 설정을 그대로 사용합니다.


## 1. 파일 불러오기 및 학습·검증 텐서 준비
SGD와 AdamW의 학습 결과를 비교하기 위해 다음 과정을 거칩니다.

- `points.csv`를 읽고 `split` 열에 따라 학습용 `train`과 검증용 `valid`를 구분합니다.
- 입력 `X`는 `x1`, `x2` 두 특성을 가진 **2차원 텐서 `(샘플 수, 2)`**이며 `float32`로 만듭니다.
- 정답 `y`는 클래스 번호를 담은 **1차원 텐서 `(샘플 수,)`**이며 `long`으로 만듭니다.
- `test` 데이터는 이번 학습과 검증에 사용하지 않습니다. 검증 데이터로 설정을 고른 뒤 최종 평가에 사용할 데이터입니다.

아래 경로 코드는 저장소 루트, `01_기본필수`, `docs` 중 어느 위치에서 노트북을 실행해도 데이터 파일을 찾도록 구성했습니다. Jupyter에서는 일반적으로 `__file__`이 정의되지 않으므로 현재 작업 폴더를 기준으로 찾습니다.


In [3]:
from pathlib import Path
import pandas as pd
import torch
from torch import nn

torch.set_num_threads(1)
candidates = [
    Path.cwd() / "points.csv",
    Path.cwd().parent / "points.csv",
    Path.cwd() / "20261030" / "01_기본필수" / "points.csv",
]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("저장소 루트, 01_기본필수 또는 docs 폴더에서 실행하세요.")
df = pd.read_csv(data_path)
train = df[df["split"] == "train"]
valid = df[df["split"] == "valid"]
X_train = torch.tensor(train[["x1", "x2"]].values, dtype=torch.float32)
y_train = torch.tensor(train["label"].values, dtype=torch.long)
X_valid = torch.tensor(valid[["x1", "x2"]].values, dtype=torch.float32)
y_valid = torch.tensor(valid["label"].values, dtype=torch.long)


## 2. Optimizer가 하는 일
손실 함수는 예측이 정답에서 얼마나 벗어났는지 계산하고, `backward()`는 손실을 각 파라미터로 미분한 기울기를 계산합니다. Optimizer는 이 기울기를 이용해 가중치와 편향을 갱신합니다.

기본 SGD의 갱신식은 다음과 같습니다. 여기서 $\theta$는 학습 파라미터, $\eta$는 학습률입니다.

$$\theta_{t+1} = \theta_t - \eta \nabla_{\theta} L(\theta_t)$$

| 항목 | SGD (이번 설정) | AdamW (이번 설정) |
| --- | --- | --- |
| 갱신 방식 | 현재 기울기에 공통 학습률을 곱해 갱신 | 기울기의 1차·2차 모멘트 추정치를 이용해 파라미터별 갱신 크기를 조절 |
| 주요 설정 | `lr=0.01`, `momentum=0.0`, `weight_decay=0.0` | `lr=0.01`, `weight_decay=0.0` |
| 이번 비교의 목적 | 기본 기울기 갱신 관찰 | 적응적 갱신 관찰 |

AdamW는 가중치 감쇠를 기울기 기반 갱신과 분리하는 방식입니다. 여기서는 `weight_decay=0.0`이므로 감쇠 효과를 비교하지 않습니다. 같은 기본 설정의 Adam과 비교하면 감쇠를 제외한 갱신 방식은 같습니다.

`lr`이 너무 크면 손실이 진동하거나 발산할 수 있고, 너무 작으면 학습이 느릴 수 있습니다. 두 알고리즘에 같은 `lr`을 주는 것은 실험 조건 중 하나이며, 각 알고리즘의 최적 학습률이 같다는 뜻은 아닙니다.


## 3. 각 최적화 기법에 따른 학습 진행 및 검증
### 모델과 손실 함수
`Linear(2, 8) → ReLU → Linear(8, 2)`는 특성 2개를 받아 두 클래스의 점수 **logits**를 출력합니다. `CrossEntropyLoss`는 logits와 정수 클래스 정답을 받으므로 모델 끝에 `Softmax`를 따로 붙이지 않습니다.

### 한 번의 학습 순서
1. `model.train()`: 학습 모드로 설정합니다. Dropout, BatchNorm이 있는 모델에서 동작에 영향을 줍니다.
2. `optimizer.zero_grad()`: 이전 단계의 기울기를 지웁니다. PyTorch는 기본적으로 기울기를 누적합니다.
3. `loss_fn(model(X_train), y_train)`: 순전파로 예측과 손실을 계산합니다.
4. `loss.backward()`: 각 파라미터의 기울기를 계산합니다. 이 단계만으로 가중치가 바뀌지는 않습니다.
5. `optimizer.step()`: 기울기를 사용해 파라미터를 갱신합니다.

이번 코드는 학습 데이터 전체를 한 번에 사용하므로 **1 epoch = 1 optimizer step**이며, 총 20번 갱신합니다. `SGD`라는 이름을 사용해도 여기서는 미니배치를 무작위로 뽑지 않는 전체 배치 학습입니다.

### 비교 조건과 검증
- 각 모델 생성 직전에 `torch.manual_seed(42)`를 호출하여 같은 초기 가중치에서 시작합니다. Optimizer도 매번 새로 생성하므로 이전 학습의 상태가 이어지지 않습니다.
- 데이터, 모델 구조, 학습 횟수, 학습률, 가중치 감쇠를 맞추고 갱신 규칙을 비교합니다.
- `model.eval()`은 평가 모드 설정이고, `torch.no_grad()`는 기울기 추적을 끄는 역할입니다. 두 기능은 별개입니다.
- 이 모델에는 Dropout과 BatchNorm이 없지만 일반적인 학습·평가 패턴을 익히기 위해 모드를 명시합니다.


In [ ]:

loss_fn = nn.CrossEntropyLoss()
for name in ["SGD", "AdamW"]:
    # 후보마다 같은 초기 가중치에서 다시 시작합니다. 앞 모델을 이어 학습하지 않습니다.
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
    if name == "SGD":
        # SGD의 학습할 파라미터는 model.parameters()입니다. lr=0.01, momentum=0.0으로 만드세요.
        # 학습률은 기본값에 기대지 않고 명시하며, 두 Optimizer 모두 weight_decay=0.0으로 비교합니다.
        # >>> B01
        # TODO B01: 위 요구사항에 맞게 이 블록만 완성합니다.
        optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.0)
        # <<< B01
    else:
        # AdamW를 같은 lr=0.01로 만드세요. weight_decay=0.0을 반드시 적으세요.
        # 이번에는 감쇠 효과가 아닌 갱신 규칙을 비교합니다. 이 공통 lr이 두 알고리즘의 최적 lr이라는 뜻은 아닙니다.
        # >>> B02
        # TODO B02: 위 요구사항에 맞게 이 블록만 완성합니다.
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.0)
        # <<< B02
    for epoch in range(20):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        valid_loss = loss_fn(model(X_valid), y_valid).item()
    print(name, "20 epochs / valid loss:", round(valid_loss, 6))


SGD 20 epochs / valid loss: 0.484183
AdamW 20 epochs / valid loss: 0.307332


## 4. 결과 해석 및 추가 실험
출력되는 값은 **20회 갱신 후 검증 데이터의 교차 엔트로피 손실**입니다. 같은 검증 데이터와 손실 함수에서는 값이 작을수록 정답 클래스에 대한 예측이 더 적절했다고 볼 수 있습니다. 정확도와 손실은 서로 다른 지표이므로 손실 감소가 항상 정확도 상승을 뜻하지는 않습니다.

한 번의 결과만으로 특정 Optimizer가 항상 더 좋다고 결론 내릴 수는 없습니다. 이번 결과는 이 데이터, 초기값, 학습률, 학습 횟수에서의 비교입니다.

### 직접 확인해 볼 내용
- **학습률 변화:** `0.001`, `0.01`, `0.1`을 각각 적용하고 검증 손실을 기록합니다. 한 번에 한 조건씩 바꾸며 각 실험은 새 모델과 새 Optimizer로 시작합니다.
- **학습 횟수 변화:** 20회와 100회를 비교해 초기 수렴 속도와 충분히 학습한 뒤의 결과를 구분합니다.
- **학습·검증 손실 기록:** epoch별 두 손실을 함께 관찰합니다. 학습 손실은 줄지만 검증 손실이 계속 커진다면 과적합 가능성을 살펴봅니다. 학습 전 손실과 갱신 후 손실을 섞어 비교하지 않도록 측정 시점을 맞춥니다.
- **초기값 변화:** 여러 seed에서 비교하되, 같은 seed의 SGD와 AdamW는 같은 초기 가중치에서 시작하도록 합니다.

검증 결과로 학습률이나 학습 횟수를 선택할 수 있지만, `test` 데이터까지 보며 설정을 반복해서 고르면 최종 평가의 의미가 약해집니다.

### 자주 하는 실수 점검
- `zero_grad()`를 빠뜨려 의도치 않게 기울기가 누적되지 않았는가?
- `backward()` 다음에 `step()`을 호출했는가?
- Optimizer가 현재 비교 중인 모델의 `parameters()`를 참조하는가?
- `CrossEntropyLoss`에 실수형 클래스 번호나 이미 Softmax를 적용한 값을 넣지 않았는가?
- AdamW의 감쇠를 끄려는 실험에서 `weight_decay=0.0`을 명시했는가?
